# Análisis Exploratorio y Limpieza — Catálogo de Series StreamView Analytics

Este notebook replica la metodología utilizada para películas, adaptada al dataset `netflix_tv_shows_detailed_up_to_2025.csv`.

**Objetivo:** identificar patrones de popularidad, calificación y géneros en las series, documentando además la calidad y las limitaciones de la fuente.

## 1. Configuración del entorno y carga de datos

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize']=(9,5.5)
ruta_datos='../data/raw/netflix_tv_shows_detailed_up_to_2025.csv'
df=pd.read_csv(ruta_datos, na_values=['','NULL','null','N/A','n/a','NaN'])
print(f'Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas')
df.head()

## 2. Diccionario de variables clave

`show_id`: identificador; `type`: tipo de contenido; `title`: título; `director`: director; `cast`: reparto; `country`: país(es); `date_added`: fecha de incorporación; `release_year`: año de estreno; `rating`: duplicado de `vote_average`; `duration`: duración; `genres`: género(s); `language`: idioma; `description`: sinopsis; `popularity`: índice de popularidad; `vote_count`: cantidad de votos; `vote_average`: calificación promedio.

## 3. Inspección de datos

In [ ]:
df.info()
print('\nValores nulos por columna:')
print(df.isnull().sum())
print('\nDuplicados exactos:', df.duplicated().sum())
print('\nResumen estadístico:')
display(df.describe(include='all').T)

### Hallazgos de calidad

- El dataset contiene 16.000 series y 16 columnas.
- No hay duplicados exactos.
- `rating` es idéntica a `vote_average` en el 100% de los registros y no aporta información adicional.
- `duration` contiene `1 Seasons` en el 100% de los registros, por lo que no permite diferenciar duración y se considera no informativa para este análisis.
- Existen valores faltantes en director, descripción, país, reparto y géneros.
- Hay 3.674 registros con `vote_count = 0`; para análisis de calificación se excluirán porque una calificación cero puede representar ausencia de votos.

## 4. Limpieza de datos

In [ ]:
df_clean=df.copy()
df_clean=df_clean.drop(columns=['rating','duration'], errors='ignore')
df_clean['date_added']=pd.to_datetime(df_clean['date_added'], errors='coerce')
df_clean['is_duplicate_title_year']=df_clean.duplicated(['title','release_year'], keep=False)
print('Columnas eliminadas: rating, duration')
print('Duplicados título+año detectados:', int(df_clean['is_duplicate_title_year'].sum()))
print('Registros finales:', len(df_clean))

## 5. Distribución de variables numéricas

In [ ]:
fig,axs=plt.subplots(1,3,figsize=(14,4.5))
for ax,col in zip(axs,['popularity','vote_count','vote_average']):
    ax.hist(df_clean[col].dropna(),bins=40); ax.set_title(col); ax.set_xlabel(col); ax.set_ylabel('Frecuencia')
plt.tight_layout(); plt.show()

## 6. Distribución de popularidad

In [ ]:
sns.kdeplot(df_clean['popularity'].dropna(), fill=True)
plt.title('Distribución de popularity — Series'); plt.xlabel('Popularity'); plt.show()

### Interpretación
`popularity` presenta una distribución asimétrica: la mayoría de los títulos se concentra en valores relativamente bajos y una cantidad pequeña alcanza valores muy altos. Esto justifica complementar el promedio con medianas y revisar outliers.

## 7. Análisis por género

In [ ]:
df_gen=df_clean.copy()
df_gen['genres']=df_gen['genres'].str.split(',')
df_gen=df_gen.explode('genres')
df_gen['genres']=df_gen['genres'].str.strip()
pop_por_genero=df_gen.groupby('genres')['popularity'].mean().sort_values(ascending=False)
pop_por_genero.head(10)

In [ ]:
pop_por_genero.head(10).sort_values().plot(kind='barh')
plt.title('Top 10 géneros por popularidad promedio — Series'); plt.xlabel('Popularidad promedio'); plt.show()

### Hallazgo
En la fuente analizada, los géneros con mayor popularidad promedio incluyen Soap, Talk, News, Western y Family. Estos resultados describen el catálogo disponible y deben interpretarse considerando que una serie puede pertenecer a varios géneros.

## 8. Volumen del catálogo por género

In [ ]:
volumen_genero=df_gen['genres'].value_counts().head(12).sort_values()
volumen_genero.plot(kind='barh')
plt.title('Top 12 géneros por cantidad de series'); plt.xlabel('Cantidad de series'); plt.show()

## 9. Evolución por año de estreno

In [ ]:
evolucion=df_clean.groupby('release_year').size()
print(evolucion)
evolucion.plot(marker='o')
plt.title('Evolución de series por año de estreno'); plt.xlabel('Año'); plt.ylabel('Cantidad de series'); plt.show()

### Limitación detectada en la fuente
**El dataset original contiene exactamente 1.000 registros para cada año entre 2010 y 2025.** Por lo tanto, la línea de evolución es plana por construcción de la fuente. No corresponde modificar estos valores para forzar una tendencia. En el informe se debe declarar esta limitación y evitar interpretar este gráfico como una evolución real del volumen de estrenos.

## 10. Calificación por género

In [ ]:
df_con_votos=df_clean[df_clean['vote_count']>0].copy()
df_cal=df_con_votos.copy(); df_cal['genres']=df_cal['genres'].str.split(','); df_cal=df_cal.explode('genres'); df_cal['genres']=df_cal['genres'].str.strip()
calificacion_por_genero=df_cal.groupby('genres')['vote_average'].mean().sort_values(ascending=False)
calificacion_por_genero.head(10)

In [ ]:
calificacion_por_genero.head(10).sort_values().plot(kind='barh')
plt.title('Top 10 géneros por calificación promedio — Series'); plt.xlabel('Calificación promedio'); plt.show()

### Interpretación
La calificación promedio por género se calcula solo sobre series con al menos un voto. En la fuente, War & Politics, Documentary, Action & Adventure, Sci-Fi & Fantasy y Animation aparecen entre los géneros con mayores promedios.

## 11. Valores atípicos

In [ ]:
fig,axs=plt.subplots(1,3,figsize=(14,4.5))
for ax,col in zip(axs,['popularity','vote_count','vote_average']):
    ax.boxplot(df_clean[col].dropna(),vert=True); ax.set_title(col)
plt.suptitle('Detección de valores atípicos — Series'); plt.tight_layout(); plt.show()

### Decisión sobre outliers
No se eliminan automáticamente. Los valores extremos de popularidad y votos pueden corresponder a títulos de alto interés y son relevantes para el análisis. Se recomienda usar medidas robustas y comunicar la dispersión.

## 12. Correlación

In [ ]:
cols=['popularity','vote_average','vote_count']
corr=df_clean[cols].corr()
sns.heatmap(corr,annot=True,fmt='.2f',center=0,vmin=-1,vmax=1,square=True)
plt.title('Matriz de correlación — Series'); plt.show()
print(corr)

### Interpretación
La matriz permite revisar la relación lineal entre popularidad, cantidad de votos y calificación. Estas correlaciones deben interpretarse como asociaciones y no como causalidad.

## 13. Preparación para el dashboard

In [ ]:
from pathlib import Path
Path('../data/processed').mkdir(exist_ok=True)
df_clean.to_csv('../data/processed/tv_shows_clean.csv',index=False)

df_gen_votos=df_con_votos.copy()
df_gen_votos['genres']=df_gen_votos['genres'].str.split(',')
df_gen_votos=df_gen_votos.explode('genres')
df_gen_votos['genres']=df_gen_votos['genres'].str.strip()
resumen_tv=(df_gen_votos.groupby('genres').agg(popularidad_promedio=('popularity','mean'),calificacion_promedio=('vote_average','mean'),cantidad_series=('show_id','nunique')).sort_values('popularidad_promedio',ascending=False))
resumen_tv.to_csv('../data/processed/resumen_por_genero_tv_shows.csv')
print('Exportados: tv_shows_clean.csv y resumen_por_genero_tv_shows.csv')

## 14. Conclusiones

1. La fuente contiene 16.000 series, sin duplicados exactos.
2. `rating` es redundante respecto de `vote_average` y `duration` es constante, por lo que ambas se eliminan del dataset limpio.
3. Existen faltantes principalmente en director, descripción, país, reparto y géneros; no se imputan automáticamente porque hacerlo podría inventar información.
4. La popularidad presenta fuerte asimetría y valores extremos, por lo que no deben eliminarse automáticamente.
5. Para calificaciones por género se excluyen registros sin votos.
6. La evolución por año no permite identificar una tendencia porque la fuente contiene exactamente 1.000 registros por año entre 2010 y 2025. Esta limitación debe declararse en el informe.